In [1]:
# -*- coding: UTF-8 -*-
import pandas as pd
import numpy as np
import warnings
from tqdm import tqdm
from datetime import datetime, timedelta
warnings.filterwarnings("ignore")
import pymysql
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StringType, StructType, StructField,
    DoubleType, DateType
)

# 初始化SparkSession
spark = SparkSession.builder \
    .appName("MyApp") \
    .enableHiveSupport() \
    .getOrCreate()

# MySQL连接配置
SOURCE_CONFIG = {
    "host": "172.20.16.9", "port": 3306,
    "user": "etl_user", "password": "Easugar@123",
    "database": "lims_gxdyty", "charset": "utf8"
}

# ==================== 新字段类型映射 ====================
NEW_FIELD_TYPE_MAPPING = {
    "factory_code": StringType(),
    "start_end_period": StringType(),
    "season_days": DoubleType(),
    "data_date": DateType(),
    "sugar_cane_crushing": DoubleType(),
    "avg_equipment_daily_crushing": DoubleType(),
    "avg_daily_crushing": DoubleType(),
    "crushing_target": DoubleType(),
    "mechanical_harvest_incoming": DoubleType(),
    "first_press_juice_brix": DoubleType(),
    "first_press_juice_purity": DoubleType(),
    "cane_sucrose_content": DoubleType(),
    "commercial_cane_sugar": DoubleType(),
    "caneas96pol10ccs": DoubleType(),
    "cane_fiber_content": DoubleType(),
    "extraction_rate": DoubleType(),
    "production_safety_rate": DoubleType(),
    "equipment_safety_rate": DoubleType(),
    "production_demand_rate": DoubleType(),
    "special_refined_sugar": DoubleType(),
    "refined_young_sugar": DoubleType(),
    "refined_sugar": DoubleType(),
    "superior_young_sugar": DoubleType(),
    "superior_sugar": DoubleType(),
    "aa_ca_sugar": DoubleType(),
    "a1_c1_sugar": DoubleType(),
    "second_grade_sugar": DoubleType(),
    "off_grade_sugar": DoubleType(),
    "sugar_powder_head": DoubleType(),
    "white_sugar_quantity": DoubleType(),
    "golden_sugar": DoubleType(),
    "raw_sugar_production": DoubleType(),
    "stored_raw_sugar": DoubleType(),
    "dissolved_raw_sugar": DoubleType(),
    "mixed_sugar_output": DoubleType(),
    "aged_white_sugar": DoubleType(),
    "raw_sugar_wip_eq_white_sugar": DoubleType(),
    "refined_sugar_wip_eq_white_sugar": DoubleType(),
    "refined_syrup_eq_white_sugar": DoubleType(),
    "aa_sugar_yield_rate": DoubleType(),
    "mixed_sugar_yield_with_wip": DoubleType(),
    "equivalent_white_sugar_yield": DoubleType(),
    "equivalent_white_sugar_yield_with_wip": DoubleType(),
    "molasses_production": DoubleType(),
    "molasses_yield_rate": DoubleType(),
    "total_recovery_rate": DoubleType(),
    "total_recovery_rate_with_wip": DoubleType(),
    "bagasse_loss": DoubleType(),
    "filter_mud_loss": DoubleType(),
    "orange_water_loss": DoubleType(),
    "undetermined_loss": DoubleType(),
    "cane_leaves_purchase": DoubleType(),
    "bagasse_sales_quantity": DoubleType(),
    "bagasse_storage": DoubleType(),
    "bagasse_cane_ratio": DoubleType(),
    "bagasse_remain_cane_ratio": DoubleType(),
    "external_power_supply": DoubleType(),
    "steam_consumption_per_ton": DoubleType(),
    "avg_steam_consumption_per_ton": DoubleType(),
    "power_consumption_per_ton": DoubleType(),
    "water_consumption_per_ton": DoubleType(),
    "external_drainage_per_ton": DoubleType()
}

# ==================== 新字段注释映射 ====================
NEW_FIELD_COMMENT_MAPPING = {
    "factory_code": "工厂代码",
    "start_end_period": "榨季起止日期",
    "season_days": "榨季经过天数",
    "data_date": "数据日期",
    "sugar_cane_crushing": "榨蔗量",
    "avg_equipment_daily_crushing": "平均设备日榨量",
    "avg_daily_crushing": "平均日榨量",
    "crushing_target": "榨蔗目标",
    "mechanical_harvest_incoming": "机收蔗进厂量",
    "first_press_juice_brix": "初压汁锤度",
    "first_press_juice_purity": "初压汁视纯度",
    "cane_sucrose_content": "甘蔗蔗糖分",
    "commercial_cane_sugar": "商业上蔗糖分CCS",
    "caneas96pol10ccs": "CANEAS96POL10CCS",
    "cane_fiber_content": "甘蔗纤维分",
    "extraction_rate": "抽出率",
    "production_safety_rate": "生产安全率",
    "equipment_safety_rate": "设备生产安全率",
    "production_demand_rate": "产需率",
    "special_refined_sugar": "特级精制白砂糖",
    "refined_young_sugar": "精幼白砂糖",
    "refined_sugar": "精制白砂糖",
    "superior_young_sugar": "优幼白砂糖",
    "superior_sugar": "优级白砂糖",
    "aa_ca_sugar": "AA/CA级白砂糖",
    "a1_c1_sugar": "A1/C1级白砂糖",
    "second_grade_sugar": "二级白砂糖",
    "off_grade_sugar": "等外白砂糖",
    "sugar_powder_head": "糖粉糖头",
    "white_sugar_quantity": "白砂糖量",
    "golden_sugar": "金砂糖",
    "raw_sugar_production": "原糖产量",
    "stored_raw_sugar": "入库原糖",
    "dissolved_raw_sugar": "洄溶原糖",
    "mixed_sugar_output": "混合糖产量",
    "aged_white_sugar": "陈化仓白砂糖量",
    "raw_sugar_wip_eq_white_sugar": "原糖在制品等折白砂糖量",
    "refined_sugar_wip_eq_white_sugar": "精糖在制品等折白砂糖量",
    "refined_syrup_eq_white_sugar": "精糖蜜等折白砂糖",
    "aa_sugar_yield_rate": "AA级白砂糖产率",
    "mixed_sugar_yield_with_wip": "混合糖产率含在制品",
    "equivalent_white_sugar_yield": "等折白砂糖产率",
    "equivalent_white_sugar_yield_with_wip": "等折白砂糖产率含在制品",
    "molasses_production": "废糖蜜产量",
    "molasses_yield_rate": "废蜜产率",
    "total_recovery_rate": "总收回率",
    "total_recovery_rate_with_wip": "总收回率含在制品",
    "bagasse_loss": "蔗渣损失",
    "filter_mud_loss": "滤泥损失",
    "orange_water_loss": "桔水损失",
    "undetermined_loss": "未测定损失",
    "cane_leaves_purchase": "蔗叶收购量",
    "bagasse_sales_quantity": "蔗渣可销售量",
    "bagasse_storage": "蔗渣存放量",
    "bagasse_cane_ratio": "蔗渣与蔗比",
    "bagasse_remain_cane_ratio": "蔗渣剩余与蔗比",
    "external_power_supply": "供外电量",
    "steam_consumption_per_ton": "吨蔗总用汽量",
    "avg_steam_consumption_per_ton": "吨蔗平均用汽量",
    "power_consumption_per_ton": "吨蔗用电量",
    "water_consumption_per_ton": "吨蔗用水量",
    "external_drainage_per_ton": "吨蔗外排水量"
}

def get_date_range(start_date):
    """获取从起始日期到昨日的日期列表"""
    yesterday = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")
    date_range = pd.date_range(start=start_date, end=yesterday)
    return date_range.strftime("%Y-%m-%d").tolist()

def get_data_from_mysql(season_cs, day_cs):
    """从MySQL获取上榨季累计数据"""
    conn = None
    cursor = None
    try:
        conn = pymysql.connect(**SOURCE_CONFIG)
        cursor = conn.cursor()
        
        prev_season = str(int(season_cs) - 1)
        
        sql = """
with gs as (select 'FN' gs union select 'FNR' union select 'TL' union select 'CZ' union select 'NM' union select 'NMR' union select 'HT'),
zhyt as (select gongsidm,niandudm,jgts z_ts, rq z_rq from bi_ytribao_tb where niandudm=%s and zuihouyitian='是'), -- 本榨季最后一天天数
s_zhyt as (select gongsidm,niandudm,jgts sz_ts, rq sz_rq from bi_ytribao_tb where niandudm=%s and zuihouyitian='是'), -- 上榨季最后一天天数
d_ts as (select gongsidm,niandudm,jgts d_ts, rq d_rq from bi_ytribao_tb where niandudm=%s and rq = %s), -- 当天经过天数
-- 选择的日期如果大于或等于勾选的最后一天，或者选择日期对应的经过天数大于上榨季最后一天天数，那么上榨季数据也是最后一天，否则就是对应本榨季日期的经过天数
ytsj as (
select t1.GongSiDM gs, t1.NIANDUDM-1 nd, if(%s>=z_rq or d_ts>=sz_ts ,sz_ts,t1.jgts) ts, t2.rq from bi_ytribao_tb t1 
left join zhyt z on z.gongsidm = t1.gongsidm and z.niandudm = t1.niandudm
left join s_zhyt sz on sz.gongsidm = t1.gongsidm and sz.niandudm = t1.niandudm-1
left join d_ts d on d.gongsidm = t1.gongsidm and d.niandudm = t1.niandudm
left join  bi_ytribao_tb t2 on t1.GongSiDM = t2.GongSiDM and t2.jgts = if(%s>=z_rq or d_ts>=sz_ts,sz_ts,t1.jgts) and t1.NIANDUDM-1 = t2.NIANDUDM
where  t1.NIANDUDM = %s and t1.RQ = if(%s>=z_rq ,z_rq,%s)  ), -- 获取上榨季原糖日报对应的日期，用于取同一天精糖日报
jtsj as (select GongSiDM gs, NIANDUDM-1 nd, jgts ts from bi_jtribao_tb  where  NIANDUDM = %s and RQ = %s ),
yt as (
select
c.GongSiDM as '公司名称',
min_rq 开始时间,
max_rq 结束时间,
c.JGTS as 榨季经过天数,
ZuiHouDengZheLiang 最后一天在制品数,
ZHAZHELIANG as 榨蔗量,
if(SHIJIZHAZHE_SJ+ifnull(SHEBEIGUZHANGSUNSHI_SJ,0)+ifnull(ZHELIAOZUAISUNSHI_SJ,0)=0,null,ZHAZHELIANG/(SHIJIZHAZHE_SJ+ifnull(SHEBEIGUZHANGSUNSHI_SJ,0)+ifnull(ZHELIAOZUAISUNSHI_SJ,0))*24) as 平均日榨量,
if(SHIJIZHAZHE_SJ-ifnull(NongWuManZha_SJ_DZ,0)=0,null,ZHAZHELIANG/(SHIJIZHAZHE_SJ-ifnull(NongWuManZha_SJ_DZ,0))*24) as 平均设备日榨量,
a.JiHuaZhaZheLiang as 榨蔗目标,
a.JISHOULIANG as 机收蔗进厂量 ,
CYZ_CUIDU as 初压汁锤度,
CYZ_SHICHUNDU as 初压汁视纯度,
GZ_ZHETANGFEN as  甘蔗蔗糖分,
sysztfccs as 商业上蔗糖分CCS,
cnseasccs as CANEAS96POL10CCS,
GZ_XIANWEIFEN as 甘蔗纤维分,
CHOUCHULV as 抽出率,
SHENGCHANANQUANLV as 生产安全率,
SHEBEISHENGCHANANQUANLV as 设备生产安全率,
CHANXULV as 产需率,
HUNHETCHANLV as 混合糖产率,
hunhetangchanliang as 混合糖产量,
AA级白砂糖,
A1级白砂糖,
二级白砂糖,
等外白砂糖,
糖粉糖头,
金砂糖,
原糖 as 原糖产量,
白砂糖量,
R3BAISHATANG 精糖蜜等折白砂糖,
if(BST_CL=0,null,BST_AA/BST_CL*100) as AA级白砂糖产率,
BAISHATANGZL as 原糖在制品等折白砂糖量,
FTM_CHANLIANG as 废糖蜜产量,
FTM_CHANLIANG/ZHAZHELIANG * 100 as 废蜜产率,
R3BAISHATANG as 等折白砂糖产率,
R3BAISHATANG as 等折白砂糖产率在制品,
SHOUHUILV_NOZZP as 总收回率,
SHOUHUILV_Z as 总收回率含在制品,
SUNSHI_ZZ as 蔗渣损失,
SUNSHI_LN as 滤泥损失,
SUNSHI_JS as 桔水损失,
SUNSHI_WCD as 未测定损失,
a.ZHEYESHOUGOU as 蔗叶收购量,
ZZ_WAIMAILIANG as 蔗渣可销售量,
ZZ_DUIFANGLIANG as 蔗渣存放量,
ZZ_SHENGYUYZB as 蔗渣剩余与蔗比,
ZZ_YUZHEBI as  蔗渣与蔗比,
GONGWAIDIANLIANG as 供外电量,
ZHENGQIYUZHEBI*10 as 吨蔗总用汽量,
吨蔗平均用汽量,
DUNZHEHAODIANLIANG as 吨蔗用电量,
YS_SL*1000/ZHAZHELIANG as 吨蔗用水量,
FS_SL*1000/ZHAZHELIANG as 吨蔗外排水量  
from (select bi_ytribao_tb.* from bi_ytribao_tb, ytsj where gongsidm in (gs) and niandudm in (nd) and jgts in (ts)) c
left join bi_ytribao_fac_3 a on a.id=c.id 
left join (select gongsidm gs, min(rq) min_rq, max(rq) max_rq from bi_ytribao_tb where NIANDUDM = %s group by gs) r on r.gs = c.gongsidm
left join (
	select gongsidm gs, SUM(if(SHENGCHANANQUANLV=100, ZHENGQIYUZHEBI*ZHAZHELIANG,0))/SUM(if(SHENGCHANANQUANLV=100, ZHAZHELIANG,0))*10 吨蔗平均用汽量 
	from ytsj
	left join bi_ytribao_tb t on ts >= jgts and gongsidm = gs and niandudm = nd
	inner join bi_ytribao_fac_1 f on t.id = f.id 
	where NIANDUDM = %s
	group by gongsidm
) a1 on c.gongsidm = a1.gs
-- left join bi_ytribao_fac_7 b on a.id =b.id and b.type='榨季累计'
left join (
select gongsidm gs, jgts,
sum(if(LEIBIEDM='AA级白砂糖',CL_Y,null)) as AA级白砂糖,
sum(if(LEIBIEDM='A1级白砂糖',CL_Y,null)) as A1级白砂糖,
sum(if(LEIBIEDM='二级白砂糖',CL_Y,null)) as 二级白砂糖,
sum(if(LEIBIEDM='等外白砂糖',CL_Y,null)) as 等外白砂糖,
sum(if(LEIBIEDM='糖粉糖头',CL_Y,null)) as 糖粉糖头,
sum(if(LEIBIEDM not in ('原糖','低色值金砂糖','高色值金砂糖','等外金砂糖'),CL_Y,null)) as 白砂糖量,
sum(if(LEIBIEDM like '%%金砂糖%%',CL_Y,null)) as 金砂糖,
sum(if(LEIBIEDM ='原糖',CL_Y,null)) as 原糖
  from bi_ytribao_fac_6 
  inner join bi_ytribao_tb  on bi_ytribao_fac_6.id=bi_ytribao_tb.id 
  where NIANDUDM = %s
  group by gongsidm, jgts
	) tb1 on c.gongsidm = tb1.gs and c.jgts = tb1.jgts

group by GongSiDM
),
jt as (
select a.GongSiDM as '公司名称',
tb1.特级精制白砂糖,
tb1.精幼白砂糖,
tb1.精制白砂糖,
tb1.优幼白砂糖,
tb1.优级白砂糖,
tb1.CA白砂糖,
tb1.C1白砂糖,
tb1.等外白砂糖,
tb1.糖粉糖头,
tb1.白砂糖量,
d.zjljl as 入库原糖,
b.CHC_BAISHATLIANG as 陈化仓白砂糖量,
b.CTL_ZZP as 混合产糖率含在制品等折量,
b.ZZP_DENGZHEBST+b.R3_DENGZHEBST as 在制品等折白砂糖产量含在制品等,
b.ZZP_DENGZHEBST as 精糖在制品等折白砂糖量,
洄溶原糖
from (select bi_jtribao_tb.* from bi_jtribao_tb, ytsj where gongsidm in (gs) and niandudm in (nd) and bi_jtribao_tb.rq in (ytsj.rq)) a 
inner join bi_jtribao_fac_3 b on a.id = b.id 
inner join bi_jtribaogs_hx_rlj c on a.id=c.id  
left join bi_jtribaocn_hx_3_3 d on a.id=d.id and d.jb = '入库原糖'
left  join (
select  gongsidm gs, jgts,
sum(if(JB='特级精制',zjlj,null)) as 特级精制白砂糖,
sum(if(JB='精幼',zjlj,null)) as 精幼白砂糖,
sum(if(JB='精制',zjlj,null)) as 精制白砂糖,
sum(if(JB='优幼',zjlj,null)) as 优幼白砂糖,
sum(if(JB='优级',zjlj,null)) as 优级白砂糖,
sum(if(JB='CA',zjlj,null)) as CA白砂糖,
sum(if(JB='C1',zjlj,null)) as C1白砂糖,
sum(if(JB='糖粉糖头' and lb='白砂糖',zjlj,null)) - sum(if(JB='糖粉糖头' and lb='洄溶糖',zjlj,0)) as 糖粉糖头,
sum(if(JB='等外',zjlj,null)) - sum(if(JB='等外糖' and lb='洄溶糖',zjlj,0)) as 等外白砂糖,
sum(if(JB='原糖',zjlj,null)) as 洄溶原糖,
(SUM(if(lb='白砂糖',zjlj,0)) - sum(if(jb in ('等外糖','糖粉糖头') and lb = '洄溶糖', zjlj,0))) 白砂糖量
from bi_jtribaocn_hx_3_2
inner join bi_jtribao_tb on bi_jtribaocn_hx_3_2.id = bi_jtribao_tb.id 
 where NIANDUDM = %s 
 group by gongsidm, jgts) tb1 on a.gongsidm = tb1.gs and a.jgts = tb1.jgts
group by GongSiDM
)

select gs.gs, concat(开始时间,'~',CHAR(10 using utf8),结束时间) 开始结束时间, 榨季经过天数, 榨蔗量, 平均设备日榨量, 平均日榨量, 榨蔗目标, 机收蔗进厂量, 初压汁锤度, 初压汁视纯度, 甘蔗蔗糖分, 商业上蔗糖分CCS, CANEAS96POL10CCS, 甘蔗纤维分, 抽出率, 生产安全率,设备生产安全率, 产需率, 特级精制白砂糖, 精幼白砂糖, 精制白砂糖, 优幼白砂糖, 优级白砂糖+ifnull(最后一天在制品数,0) 优级白砂糖, ifnull(AA级白砂糖,CA白砂糖) 'AA/CA', ifnull(A1级白砂糖,C1白砂糖) 'A1/C1', 二级白砂糖, ifnull(yt.等外白砂糖,jt.等外白砂糖) 等外白砂糖,ifnull(yt.糖粉糖头,jt.糖粉糖头) 糖粉糖头, ifnull(yt.白砂糖量,jt.白砂糖量) 白砂糖量, 金砂糖, 原糖产量, ifnull(入库原糖,原糖产量) 入库原糖, 洄溶原糖, 
(ifnull(ifnull(yt.白砂糖量,jt.白砂糖量),0)+ifnull(金砂糖,0)+ifnull(ifnull(入库原糖,原糖产量),0)+ifnull(最后一天在制品数,0)) 混合糖产量, 陈化仓白砂糖量, 原糖在制品等折白砂糖量, round(精糖在制品等折白砂糖量,3)-ifnull(最后一天在制品数,0) 精糖在制品等折白砂糖量, 精糖蜜等折白砂糖, AA级白砂糖产率, 
(ifnull(ifnull(yt.白砂糖量,jt.白砂糖量),0)+ifnull(金砂糖,0)+ifnull(ifnull(入库原糖,原糖产量),0)+ifnull(陈化仓白砂糖量,0)+ifnull(原糖在制品等折白砂糖量,0)+ifnull(精糖在制品等折白砂糖量,0))/榨蔗量*1000 混合糖产率含在制品, 
(ifnull(ifnull(yt.白砂糖量,jt.白砂糖量),0)+ifnull(金砂糖,0)+ifnull(ifnull(入库原糖,原糖产量),0)*0.97+ifnull(最后一天在制品数,0))/榨蔗量*1000 等折白砂糖产率, 
(ifnull(ifnull(yt.白砂糖量,jt.白砂糖量),0)+ifnull(金砂糖,0)+ifnull(ifnull(入库原糖,原糖产量),0)*0.97 +ifnull(陈化仓白砂糖量,0)+ifnull(原糖在制品等折白砂糖量,0)+ifnull(精糖在制品等折白砂糖量,0))/榨蔗量*1000 等折白砂糖产率含在制品,
废糖蜜产量, 废蜜产率, 总收回率, 总收回率含在制品, 蔗渣损失, 滤泥损失, 桔水损失, 未测定损失, 蔗叶收购量, 蔗渣可销售量, 蔗渣存放量, 蔗渣与蔗比, 蔗渣剩余与蔗比, 供外电量, 吨蔗总用汽量, 吨蔗平均用汽量, 吨蔗用电量, 吨蔗用水量, 吨蔗外排水量
from gs
left join (select gs.gs,yt.* from gs left join yt on gs.gs = yt.公司名称) yt on gs.gs = yt.gs
left join (select gs.gs,jt.* from gs left join jt on gs.gs = jt.公司名称) jt on gs.gs = jt.gs
"""
        params = [
            season_cs,          # zhyt
            prev_season,        # s_zhyt
            season_cs,          # d_ts (niandudm)
            day_cs,             # d_ts (rq)
            day_cs,             # ytsj 条件1
            day_cs,             # ytsj 条件2
            season_cs,          # ytsj t1.NIANDUDM
            day_cs,             # ytsj if条件
            day_cs,             # ytsj else
            season_cs,          # jtsj NIANDUDM
            day_cs,             # jtsj RQ
            prev_season,        # yt min_rq
            prev_season,        # a1
            prev_season,        # tb1
            prev_season         # jt tb1
        ]
        
        cursor.execute(sql, params)
        result = cursor.fetchall()
        cols = [c[0] for c in cursor.description]
        df = pd.DataFrame(result, columns=cols) if result else pd.DataFrame(columns=cols)
        return df
        
    except Exception as e:
        print(f"获取数据失败{day_cs}: {e}")
        raise
    finally:
        if cursor:
            try:
                cursor.close()
            except:
                pass
        if conn:
            try:
                conn.close()
            except:
                pass

def translate_column_names(df):
    """将中文列名翻译为英文列名"""
    cn_to_en_mapping = {
        'gs': 'factory_code',
        '开始结束时间': 'start_end_period',
        '榨季经过天数': 'season_days',
        '榨蔗量': 'sugar_cane_crushing',
        '平均设备日榨量': 'avg_equipment_daily_crushing',
        '平均日榨量': 'avg_daily_crushing',
        '榨蔗目标': 'crushing_target',
        '机收蔗进厂量': 'mechanical_harvest_incoming',
        '初压汁锤度': 'first_press_juice_brix',
        '初压汁视纯度': 'first_press_juice_purity',
        '甘蔗蔗糖分': 'cane_sucrose_content',
        '商业上蔗糖分CCS': 'commercial_cane_sugar',
        'CANEAS96POL10CCS': 'caneas96pol10ccs',
        '甘蔗纤维分': 'cane_fiber_content',
        '抽出率': 'extraction_rate',
        '生产安全率': 'production_safety_rate',
        '设备生产安全率': 'equipment_safety_rate',
        '产需率': 'production_demand_rate',
        '特级精制白砂糖': 'special_refined_sugar',
        '精幼白砂糖': 'refined_young_sugar',
        '精制白砂糖': 'refined_sugar',
        '优幼白砂糖': 'superior_young_sugar',
        '优级白砂糖': 'superior_sugar',
        'AA/CA': 'aa_ca_sugar',
        'A1/C1': 'a1_c1_sugar',
        '二级白砂糖': 'second_grade_sugar',
        '等外白砂糖': 'off_grade_sugar',
        '糖粉糖头': 'sugar_powder_head',
        '白砂糖量': 'white_sugar_quantity',
        '金砂糖': 'golden_sugar',
        '原糖产量': 'raw_sugar_production',
        '入库原糖': 'stored_raw_sugar',
        '洄溶原糖': 'dissolved_raw_sugar',
        '混合糖产量': 'mixed_sugar_output',
        '陈化仓白砂糖量': 'aged_white_sugar',
        '原糖在制品等折白砂糖量': 'raw_sugar_wip_eq_white_sugar',
        '精糖在制品等折白砂糖量': 'refined_sugar_wip_eq_white_sugar',
        '精糖蜜等折白砂糖': 'refined_syrup_eq_white_sugar',
        'AA级白砂糖产率': 'aa_sugar_yield_rate',
        '混合糖产率含在制品': 'mixed_sugar_yield_with_wip',
        '等折白砂糖产率': 'equivalent_white_sugar_yield',
        '等折白砂糖产率含在制品': 'equivalent_white_sugar_yield_with_wip',
        '废糖蜜产量': 'molasses_production',
        '废蜜产率': 'molasses_yield_rate',
        '总收回率': 'total_recovery_rate',
        '总收回率含在制品': 'total_recovery_rate_with_wip',
        '蔗渣损失': 'bagasse_loss',
        '滤泥损失': 'filter_mud_loss',
        '桔水损失': 'orange_water_loss',
        '未测定损失': 'undetermined_loss',
        '蔗叶收购量': 'cane_leaves_purchase',
        '蔗渣可销售量': 'bagasse_sales_quantity',
        '蔗渣存放量': 'bagasse_storage',
        '蔗渣与蔗比': 'bagasse_cane_ratio',
        '蔗渣剩余与蔗比': 'bagasse_remain_cane_ratio',
        '供外电量': 'external_power_supply',
        '吨蔗总用汽量': 'steam_consumption_per_ton',
        '吨蔗平均用汽量': 'avg_steam_consumption_per_ton',
        '吨蔗用电量': 'power_consumption_per_ton',
        '吨蔗用水量': 'water_consumption_per_ton',
        '吨蔗外排水量': 'external_drainage_per_ton'
    }
    existing_rename = {cn: en for cn, en in cn_to_en_mapping.items() if cn in df.columns}
    df_renamed = df.rename(columns=existing_rename)
    return df_renamed, NEW_FIELD_COMMENT_MAPPING

def clean_data_columns(pandas_df):
    """清理数据列，标准化空值"""
    if pandas_df.empty:
        return pandas_df
    
    string_cols = ["factory_code", "start_end_period"]
    date_cols = ["data_date"]
    numeric_cols = [col for col in NEW_FIELD_TYPE_MAPPING.keys() if col not in string_cols + date_cols]
    string_cols = [col for col in string_cols if col in pandas_df.columns]
    date_cols = [col for col in date_cols if col in pandas_df.columns]
    numeric_cols = [col for col in numeric_cols if col in pandas_df.columns]

    null_markers = [None, 'None', 'NULL', 'null', 'NaN', 'nan', '', ' ', '—', '·']
    for col in pandas_df.columns:
        pandas_df[col] = pandas_df[col].replace(null_markers, pd.NA)

    for col in string_cols:
        pandas_df[col] = pandas_df[col].fillna('').astype(str).str.strip()
        if col == "factory_code":
            pandas_df[col] = pandas_df[col].str[:50]
        else:
            pandas_df[col] = pandas_df[col].str[:100]

    for col in date_cols:
        pandas_df[col] = pd.to_datetime(pandas_df[col], errors='coerce')

    for col in numeric_cols:
        if pandas_df[col].dtype == 'object':
            pandas_df[col] = pd.to_numeric(pandas_df[col], errors='coerce')
        pandas_df[col] = pandas_df[col].astype('float').round(4).fillna(0.0)

    return pandas_df.reset_index(drop=True)

def create_spark_df(pandas_df):
    """创建Spark DataFrame"""
    pandas_df_clean = clean_data_columns(pandas_df)
    if pandas_df_clean.empty:
        raise ValueError("清理后的数据为空，无法创建Spark DataFrame")
    
    existing_cols = [col for col in pandas_df_clean.columns if col in NEW_FIELD_TYPE_MAPPING.keys()]
    
    schema_fields = []
    for col in existing_cols:
        field_type = NEW_FIELD_TYPE_MAPPING.get(col, StringType())
        schema_fields.append(StructField(col, field_type, nullable=True))
    schema = StructType(schema_fields)
    
    for col in existing_cols:
        if isinstance(NEW_FIELD_TYPE_MAPPING.get(col), DateType) and col in pandas_df_clean.columns:
            pandas_df_clean[col] = pd.to_datetime(pandas_df_clean[col], errors='coerce')
    
    spark_df = spark.createDataFrame(pandas_df_clean[existing_cols], schema=schema)
    return spark_df

def write_to_hive_with_comments(spark_df, target_table, column_comments):
    """写入Hive表"""
    try:
        if spark_df.count() == 0:
            print("Spark DataFrame为空，跳过写入Hive")
            return
        
        spark_df.createOrReplaceTempView("temp_view")
        
        columns_sql = []
        for field in spark_df.schema.fields:
            col_name = field.name
            spark_type = field.dataType.simpleString()
            if spark_type == "double":
                hive_type = "DOUBLE"
            elif spark_type == "date":
                hive_type = "DATE"
            else:
                hive_type = "STRING"
            comment = column_comments.get(col_name, "").replace("'", "''")
            columns_sql.append(f"`{col_name}` {hive_type} COMMENT '{comment}'" if comment else f"`{col_name}` {hive_type}")
        
        spark.sql(f"DROP TABLE IF EXISTS {target_table}")
        create_sql = f"""
        CREATE TABLE {target_table} (
            {', '.join(columns_sql)}
        ) 
        STORED AS ORC
        COMMENT '上榨季累计数据表（日粒度）'
        """
        spark.sql(create_sql)
        
        spark.sql(f"INSERT INTO {target_table} SELECT * FROM temp_view")
        print(f"写入Hive成功: {target_table}, 行数: {spark_df.count()}")
        
    except Exception as e:
        print(f"写入Hive失败: {e}")
        raise

def process_all_dates(season_cs, start_date):
    """处理所有日期的数据"""
    date_list = get_date_range(start_date)
    all_data_frames = []
    
    for day_cs in tqdm(date_list, desc="处理进度"):
        try:
            pandas_df = get_data_from_mysql(season_cs, day_cs)
            if not pandas_df.empty:
                pandas_df['data_date'] = day_cs
                all_data_frames.append(pandas_df)
        except Exception as e:
            print(f"跳过日期{day_cs}：{e}")
            continue
    
    if all_data_frames:
        combined_df = pd.concat(all_data_frames, ignore_index=True)
        return combined_df
    return pd.DataFrame()

if __name__ == "__main__":
    season_cs = "2026"
    start_date = "2025-11-30"
    
    print(f"开始处理: {start_date} 至昨日")
    
    # 1. 获取合并数据
    combined_pandas_df = process_all_dates(season_cs, start_date)
    if combined_pandas_df.empty:
        print("无有效数据，任务结束")
        spark.stop()
        exit(0)
    
    # 2. 列名翻译
    pandas_df_renamed, column_comments = translate_column_names(combined_pandas_df)
    
    # 3. 打印调试信息
    print(f"\n实际获取的列数: {len(pandas_df_renamed.columns)}")
    print(f"实际列名: {list(pandas_df_renamed.columns)[:5]}...（省略部分列）")
    print("\n数据样例（前3行，部分列）- 空值已处理:")
    print(pandas_df_renamed[['factory_code', 'data_date', 'sugar_cane_crushing']].head(3))
    
    # 4. 创建Spark DataFrame
    spark_df = create_spark_df(pandas_df_renamed)
    
    # 5. 打印Spark Schema
    print("\nSpark DataFrame Schema:")
    spark_df.printSchema()
    
    # 6. 写入Hive表
    target_table = "app.app_lims_prev_season_cumulative_f_1d"
    write_to_hive_with_comments(spark_df, target_table, column_comments)
    
    # 7. 关闭Spark会话
    spark.stop()
    print("\n任务完成")

开始处理: 2025-11-30 至昨日


处理进度: 100%|██████████| 117/117 [00:40<00:00,  2.87it/s]



实际获取的列数: 62
实际列名: ['factory_code', 'start_end_period', 'season_days', 'sugar_cane_crushing', 'avg_equipment_daily_crushing']...（省略部分列）

数据样例（前3行，部分列）- 空值已处理:
  factory_code   data_date sugar_cane_crushing
0           FN  2025-11-30     6794.1470000000
1          FNR  2025-11-30                None
2           TL  2025-11-30                None

Spark DataFrame Schema:
root
 |-- factory_code: string (nullable = true)
 |-- start_end_period: string (nullable = true)
 |-- season_days: double (nullable = true)
 |-- sugar_cane_crushing: double (nullable = true)
 |-- avg_equipment_daily_crushing: double (nullable = true)
 |-- avg_daily_crushing: double (nullable = true)
 |-- crushing_target: double (nullable = true)
 |-- mechanical_harvest_incoming: double (nullable = true)
 |-- first_press_juice_brix: double (nullable = true)
 |-- first_press_juice_purity: double (nullable = true)
 |-- cane_sucrose_content: double (nullable = true)
 |-- commercial_cane_sugar: double (nullable = true)
 |-- c